# 意图识别功能测试

测试 `intent_node` 的意图分类和字段抽取能力，覆盖以下场景：
- 完整 / 不完整报修信息
- 多种服务类型（维修、安装、测量）
- 确认 / 取消订单
- 闲聊 / 未知意图
- 多轮对话上下文感知

## 1. 环境初始化

In [17]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import asyncio
import json
import nest_asyncio
from collections import defaultdict
from rich.console import Console
from rich.table import Table
from rich import box

nest_asyncio.apply()

from langchain_core.messages import HumanMessage, AIMessage
from graph.builder import intent_node
from graph.state import AgentState

console = Console()
print('✅ 环境初始化完成')

✅ 环境初始化完成


## 2. 测试用例定义

In [18]:
TEST_CASES = [
    # ── 基础下单 ──────────────────────────────────────────────
    {
        "id": "TC01",
        "group": "下单-完整",
        "desc": "完整报修：房号+设备+故障+紧急度",
        "user_input": "1208房间空调不制冷，比较急",
        "status": "idle",
        "expected": {
            "intent": "create_order",
            "room_number": "1208",
            "product": "空调",
            "fault": "不制冷",
            "urgency": "high",
        },
    },
    {
        "id": "TC02",
        "group": "下单-完整",
        "desc": "水龙头漏水，包含区域",
        "user_input": "0316卫生间水龙头漏水",
        "status": "idle",
        "expected": {
            "intent": "create_order",
            "room_number": "0316",
            "product": "水龙头",
            "fault": "漏水",
            "area": "卫生间",
        },
    },
    # ── 下单-缺字段 ───────────────────────────────────────────
    {
        "id": "TC03",
        "group": "下单-缺字段",
        "desc": "只有设备和故障，缺房号",
        "user_input": "空调不制冷",
        "status": "idle",
        "expected": {
            "intent": "create_order",
            "product": "空调",
            "fault": "不制冷",
            "room_number": None,
        },
    },
    {
        "id": "TC04",
        "group": "下单-缺字段",
        "desc": "只说房号，缺设备和故障",
        "user_input": "我在1503房",
        "status": "collecting",
        "expected": {
            "intent": "create_order",
            "room_number": "1503",
            "product": None,
            "fault": None,
        },
    },
    # ── 服务类型 ──────────────────────────────────────────────
    {
        "id": "TC05",
        "group": "服务类型",
        "desc": "单次安装",
        "user_input": "0801房要安装一个浴霸",
        "status": "idle",
        "expected": {
            "intent": "create_order",
            "service_type": "单次安装",
            "room_number": "0801",
        },
    },
    {
        "id": "TC06",
        "group": "服务类型",
        "desc": "单次测量",
        "user_input": "需要测量一下1102房窗帘的尺寸",
        "status": "idle",
        "expected": {
            "intent": "create_order",
            "service_type": "单次测量",
        },
    },
    # ── 确认 / 取消 ───────────────────────────────────────────
    {
        "id": "TC07",
        "group": "确认/取消",
        "desc": "用户明确确认提交",
        "user_input": "确认提交",
        "status": "confirming",
        "prior_messages": [
            ("human", "1208空调不制冷"),
            ("ai", "已收集到信息，请确认：1208房间，空调不制冷。"),
        ],
        "expected": {
            "intent": "confirm_order",
            "user_confirmed": True,
        },
    },
    {
        "id": "TC08",
        "group": "确认/取消",
        "desc": "没问题=确认提交",
        "user_input": "没问题",
        "status": "confirming",
        "prior_messages": [
            ("human", "空调坏了1208"),
            ("ai", "请确认订单信息是否正确？"),
        ],
        "expected": {
            "intent": "confirm_order",
            "user_confirmed": True,
        },
    },
    {
        "id": "TC09",
        "group": "确认/取消",
        "desc": "取消当前订单",
        "user_input": "不用了",
        "status": "collecting",
        "expected": {
            "intent": "cancel_order",
            "user_cancelled": True,
        },
    },
    {
        "id": "TC10",
        "group": "确认/取消",
        "desc": "取消-先算了",
        "user_input": "先算了",
        "status": "confirming",
        "expected": {
            "intent": "cancel_order",
            "user_cancelled": True,
        },
    },
    # ── 闲聊 / 未知 ───────────────────────────────────────────
    {
        "id": "TC11",
        "group": "闲聊/未知",
        "desc": "问天气（闲聊）",
        "user_input": "今天天气怎么样",
        "status": "idle",
        "expected": {"intent": "smalltalk"},
    },
    {
        "id": "TC12",
        "group": "闲聊/未知",
        "desc": "问Wi-Fi密码（闲聊）",
        "user_input": "Wi-Fi密码是多少",
        "status": "idle",
        "expected": {"intent": "smalltalk"},
    },
    {
        "id": "TC13",
        "group": "闲聊/未知",
        "desc": "无意义输入",
        "user_input": "嗯嗯好的",
        "status": "idle",
        "expected": {"intent": "unknown"},
    },
    # ── 上下文感知 ────────────────────────────────────────────
    {
        "id": "TC14",
        "group": "上下文感知",
        "desc": "补充房号（多轮对话中）",
        "user_input": "1208",
        "status": "collecting",
        "prior_messages": [
            ("human", "空调不制冷"),
            ("ai", "请问您在哪个房间？"),
        ],
        "order_info": {"product": "空调", "fault": "不制冷"},
        "expected": {
            "intent": "create_order",
            "room_number": "1208",
        },
    },
    {
        "id": "TC15",
        "group": "上下文感知",
        "desc": "已提交订单后闲聊，不复用历史字段",
        "user_input": "谢谢",
        "status": "submitted",
        "last_order": {"room_number": "1208", "product": "空调", "fault": "不制冷"},
        "expected": {
            "intent": "smalltalk",
            "room_number": None,
            "product": None,
        },
    },
    # ── 区域别名 ──────────────────────────────────────────────
    {
        "id": "TC16",
        "group": "字段归一",
        "desc": "厕所→卫生间",
        "user_input": "0520厕所马桶堵塞",
        "status": "idle",
        "expected": {
            "intent": "create_order",
            "area": "卫生间",
            "product": "马桶",
            "fault": "堵塞",
        },
    },
    {
        "id": "TC17",
        "group": "字段归一",
        "desc": "紧急度：马上来=urgent",
        "user_input": "1102漏电了马上来",
        "status": "idle",
        "expected": {
            "intent": "create_order",
            "urgency": "urgent",
        },
    },
]

print(f'已定义 {len(TEST_CASES)} 个测试用例，共 {len(set(tc["group"] for tc in TEST_CASES))} 个分组')

已定义 17 个测试用例，共 7 个分组


## 3. 测试执行器

In [19]:
def build_state(tc: dict) -> AgentState:
    """从测试用例构造 AgentState。"""
    messages = []
    for role, content in tc.get("prior_messages", []):
        messages.append(HumanMessage(content=content) if role == "human" else AIMessage(content=content))
    messages.append(HumanMessage(content=tc["user_input"]))

    return AgentState(
        messages=messages,
        status=tc.get("status", "idle"),
        order_info=tc.get("order_info", {}),
        last_order=tc.get("last_order", {}),
        last_user_message=tc["user_input"],
    )


def check_fields(result: dict, expected: dict) -> tuple[list[str], list[str]]:
    """返回 (通过的字段, 失败的字段)。"""
    passed, failed = [], []
    order_info = result.get("order_info", {})

    for field, exp_val in expected.items():
        if field == "intent":
            actual = result.get("intent")
        elif field in ("user_confirmed", "user_cancelled"):
            actual = order_info.get(field)
        elif field == "service_type":
            actual = result.get("service_type")
        else:
            actual = order_info.get(field)

        if actual == exp_val:
            passed.append(field)
        else:
            failed.append(f"{field}(期望={exp_val!r}, 实际={actual!r})")

    return passed, failed


async def run_test_case(tc: dict) -> dict:
    """执行单个测试用例，返回结构化结果。"""
    state = build_state(tc)
    try:
        result = await intent_node(state)
        passed, failed = check_fields(result, tc["expected"])
        return {
            "id": tc["id"],
            "group": tc["group"],
            "desc": tc["desc"],
            "user_input": tc["user_input"],
            "intent": result.get("intent"),
            "service_type": result.get("service_type"),
            "order_info": result.get("order_info", {}),
            "passed": passed,
            "failed": failed,
            "status": "✅ PASS" if not failed else "❌ FAIL",
            "error": None,
        }
    except Exception as e:
        return {
            "id": tc["id"],
            "group": tc["group"],
            "desc": tc["desc"],
            "user_input": tc["user_input"],
            "intent": None,
            "service_type": None,
            "order_info": {},
            "passed": [],
            "failed": [],
            "status": "💥 ERROR",
            "error": str(e),
        }

print('测试执行器已就绪')

测试执行器已就绪


## 4. 单条用例快速验证

In [20]:
# 修改 tc_id 测试任意用例
tc_id = "TC01"

tc = next(t for t in TEST_CASES if t["id"] == tc_id)
result = asyncio.run(run_test_case(tc))

print(f"{'='*55}")
print(f"[{result['id']}] {result['desc']}")
print(f"输入  : {result['user_input']}")
print(f"意图  : {result['intent']}")
print(f"服务类型: {result['service_type']}")
print(f"订单信息: {json.dumps(result['order_info'], ensure_ascii=False, indent=2)}")
print(f"结果  : {result['status']}")
if result['failed']:
    print(f"失败字段: {result['failed']}")
if result['error']:
    print(f"异常  : {result['error']}")

[TC01] 完整报修：房号+设备+故障+紧急度
输入  : 1208房间空调不制冷，比较急
意图  : create_order
服务类型: 单次维修服务
订单信息: {
  "room_number": "1208",
  "product": "空调",
  "fault": "不制冷",
  "urgency": "high",
  "user_confirmed": false,
  "user_cancelled": false
}
结果  : ✅ PASS


## 5. 全量批量测试

In [21]:
async def run_all(cases: list[dict]) -> list[dict]:
    tasks = [run_test_case(tc) for tc in cases]
    return await asyncio.gather(*tasks)

print(f'正在运行 {len(TEST_CASES)} 个用例（并发）...')
all_results = asyncio.run(run_all(TEST_CASES))
print('完成')

正在运行 17 个用例（并发）...


KeyboardInterrupt: 

## 6. 结果汇总

In [ ]:
pass_count  = sum(1 for r in all_results if r["status"] == "✅ PASS")
fail_count  = sum(1 for r in all_results if r["status"] == "❌ FAIL")
error_count = sum(1 for r in all_results if r["status"] == "💥 ERROR")
total = len(all_results)

console.print(f"\n[bold]通过: {pass_count}/{total}  失败: {fail_count}  异常: {error_count}  通过率: {pass_count/total:.0%}[/bold]\n")

# 主结果表
t = Table(box=box.ROUNDED, show_lines=True, highlight=True)
for col in ["ID", "分组", "描述", "输入", "意图", "服务类型", "房号", "设备", "故障", "区域", "紧急度", "结果"]:
    t.add_column(col, overflow="fold")

for r in all_results:
    oi = r["order_info"]
    style = "green" if r["status"] == "✅ PASS" else ("red" if r["status"] == "❌ FAIL" else "yellow")
    fail_note = ("\n[dim]" + " | ".join(r["failed"]) + "[/dim]") if r["failed"] else ""
    t.add_row(
        r["id"], r["group"], r["desc"], r["user_input"],
        r["intent"] or "-", r["service_type"] or "-",
        oi.get("room_number") or "-", oi.get("product") or "-",
        oi.get("fault") or "-", oi.get("area") or "-",
        oi.get("urgency") or "-",
        f"[{style}]{r['status']}[/{style}]{fail_note}",
    )

console.print(t)

## 7. 分组通过率

In [ ]:
group_data = defaultdict(lambda: {"总数": 0, "通过": 0, "失败": 0, "异常": 0})
for r in all_results:
    g = group_data[r["group"]]
    g["总数"] += 1
    if r["status"] == "✅ PASS":   g["通过"] += 1
    elif r["status"] == "❌ FAIL": g["失败"] += 1
    else:                           g["异常"] += 1

gt = Table(title="分组通过率", box=box.ROUNDED)
for col in ["分组", "总数", "通过", "失败", "异常", "通过率"]:
    gt.add_column(col, justify="center")

for group, s in sorted(group_data.items()):
    rate = s["通过"] / s["总数"]
    color = "green" if rate == 1.0 else ("yellow" if rate >= 0.5 else "red")
    gt.add_row(
        group, str(s["总数"]),
        f"[green]{s['通过']}[/green]",
        f"[red]{s['失败']}[/red]" if s["失败"] else "0",
        f"[yellow]{s['异常']}[/yellow]" if s["异常"] else "0",
        f"[{color}]{rate:.0%}[/{color}]",
    )

console.print(gt)

## 8. 失败用例详情

In [ ]:
failed_cases = [r for r in all_results if r["status"] != "✅ PASS"]

if not failed_cases:
    print("🎉 所有用例全部通过！")
else:
    for r in failed_cases:
        print(f"{'─'*55}")
        print(f"[{r['id']}] {r['desc']}  {r['status']}")
        print(f"  输入    : {r['user_input']}")
        print(f"  识别意图 : {r['intent']}")
        print(f"  订单信息 : {json.dumps(r['order_info'], ensure_ascii=False)}")
        if r["failed"]:
            print(f"  失败字段 : {r['failed']}")
        if r["error"]:
            print(f"  异常    : {r['error']}")

## 9. 自由探索

In [23]:
async def quick_test(user_input: str, status: str = "idle", order_info: dict | None = None):
    """快速测试单条输入，不需要预期值。"""
    state = AgentState(
        messages=[HumanMessage(content=user_input)],
        status=status,
        order_info=order_info or {},
        last_order={},
        last_user_message=user_input,
    )
    result = await intent_node(state)
    print(f"输入    : {user_input}")
    print(f"意图    : {result.get('intent')}")
    print(f"服务类型 : {result.get('service_type')}")
    print(f"订单信息 : {json.dumps(result.get('order_info', {}), ensure_ascii=False, indent=2)}")

# 在这里随意输入测试
asyncio.run(quick_test("B栋502洗碗机不通电，不急"))

输入    : B栋502洗碗机不通电，不急
意图    : create_order
服务类型 : 单次维修服务
订单信息 : {
  "room_number": "B 栋 502",
  "product": "洗碗机",
  "fault": "不通电",
  "urgency": "low",
  "user_confirmed": false,
  "user_cancelled": false
}
